# 17 — Case 3: Self-Supervised Brain-HRF Co-Embedding

Case 3 pairs each brain window with *its own window's* HRF regressor vector
(same-window co-occurrence) rather than a class-label-derived target. No
class label enters the loss anywhere, which makes this genuinely
**self-supervised** — a different learning paradigm from Case 1 (supervised
multi-task decoding) and Case 2 (supervised contrastive, SupCon-style
against label-derived text prototypes). See
[`docs/case2-3-design-plan.md`](../docs/case2-3-design-plan.md) §2.5 for
the full design rationale and disambiguation from the earlier, unrelated
SLDS/rSLDS idea (not "Case 3" — see §0 there).

Because Case 3 has no classifier head or class-conditional output by
construction (it never sees a label), this notebook also validates the
`BrainWithPostHocClassifier` wrapper: a linear probe fit post-hoc on frozen
Case-3 features, giving (a) a macro-F1 number directly comparable to Case 1
(92.0%) and Case 2 (91.8%), and (b) a target-class logit for CAV/TCAV to
differentiate, letting Case 1's existing `concepts.py` machinery run on
Case 3 completely unchanged.

**Scope of this notebook**: validates the full pipeline (train -> post-hoc
probe -> fixed 5-concept CAV/TCAV, with the cross-class rank-bootstrap
significance test from `population-level-evaluation-plan.md` §6) on the
**current 100-subject data**, per the project's established
prototype-then-scale workflow. The full 3-case x 2-architecture x
capacity-variant sweep on the 200-subject dataset is a separate, later
step (task #58), sequenced after the data scale-up completes.

In [1]:
from __future__ import annotations

import json
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "environment.yml").exists() or (candidate / ".git").exists():
            return candidate
    raise FileNotFoundError("Could not locate the NeuroLens repository root.")


PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from neurolens.data_setup import make_dataloaders
from neurolens.model_builder import GRUDecoder, TransformerDecoder
from neurolens.engine import get_device
from neurolens.case3 import BrainHRFModel, train_case3, fit_post_hoc_classifier
from neurolens.concepts import (
    CONCEPT_DEFINITIONS,
    extract_pooled_features,
    extract_pooled_features_with_subjects,
    train_cav,
    run_concept_analysis,
    cross_class_rank_bootstrap_test,
)

PROCESSED_ROOT = PROJECT_ROOT / "data" / "processed" / "hcp_ya_s1200" / "runs"
RESULTS_DIR = PROJECT_ROOT / "results"
MODELS_DIR = PROJECT_ROOT / "models"

SEED = 42
EMBED_DIM = 64
NUM_EPOCHS = 5


def set_seed(seed: int) -> None:
    torch.manual_seed(seed)
    np.random.seed(seed)


device = get_device()
print("device:", device)

device: mps


## Part A — Primary training run: GRU vs. Transformer backbones

In [2]:
train_loader, val_loader, test_loader, info = make_dataloaders(PROCESSED_ROOT, batch_size=64, window_length=32)
class_names = [info["class_to_condition"][str(c)] for c in range(info["num_classes"])]
print("classes:", class_names)
print("n_train windows:", len(train_loader.dataset), "n_test windows:", len(test_loader.dataset))

case3_models = {}
case3_primary_results = {}

for arch_name, backbone_fn, backbone_dim in [
    ("GRU", lambda: GRUDecoder(num_classes=info["num_classes"], num_conditions=info["num_conditions"], include_hrf_head=False), 128),
    ("Transformer", lambda: TransformerDecoder(num_classes=info["num_classes"], num_conditions=info["num_conditions"], include_hrf_head=False), 128),
]:
    set_seed(SEED)
    backbone = backbone_fn()
    model = BrainHRFModel(backbone, backbone_dim=backbone_dim, embed_dim=EMBED_DIM)
    t0 = time.time()
    train_result = train_case3(
        model, train_loader, val_loader, device,
        num_epochs=NUM_EPOCHS, experiment_name=f"case3_{arch_name.lower()}",
        checkpoint_dir=MODELS_DIR / f"case3_{arch_name.lower()}",
    )
    wrapped, metrics = fit_post_hoc_classifier(model, train_loader, test_loader, device, num_classes=info["num_classes"])
    elapsed = time.time() - t0
    print(f"[{arch_name}] best_val_contrastive_loss={train_result['best_val_loss']:.4f} "
          f"post_hoc_test_macro_f1={metrics['macro_f1']:.4f} ({elapsed:.0f}s)")
    case3_models[arch_name] = {"contrastive_model": model, "wrapped": wrapped}
    case3_primary_results[arch_name] = {"best_val_contrastive_loss": train_result["best_val_loss"], "test_metrics": metrics}

print()
print("(Case 1 Transformer multi-task test macro F1 was 0.920, Case 2 was 0.918, for comparison)")

classes: ['baseline', 'left_hand', 'right_hand', 'left_foot', 'right_foot', 'tongue']
n_train windows: 16510 n_test windows: 3048


[case3_gru] epoch 1/5 train_loss=2.7088 val_loss=2.3617


[case3_gru] epoch 2/5 train_loss=1.9088 val_loss=2.0521


[case3_gru] epoch 3/5 train_loss=1.3682 val_loss=1.8537


[case3_gru] epoch 4/5 train_loss=0.9969 val_loss=1.7760


...

[case3_transformer] epoch 2/5 train_loss=1.6468 val_loss=1.8818


[case3_transformer] epoch 3/5 train_loss=1.2633 val_loss=1.6298


[case3_transformer] epoch 4/5 train_loss=1.0112 val_loss=1.5117


[case3_transformer] epoch 5/5 train_loss=0.8897 val_loss=1.4678


[Transformer] best_val_contrastive_loss=1.4678 post_hoc_test_macro_f1=0.9171 (16s)

(Case 1 Transformer multi-task test macro F1 was 0.920, Case 2 was 0.918, for comparison)


## Part B — Fixed (closed-vocabulary) concept testing

Case 3 has no text branch, so it cannot support open-vocabulary CAV the way
Case 2 does — there is nothing to embed an arbitrary phrase into. Only the
5 label-derived concepts from `CONCEPT_DEFINITIONS` (hand, foot, tongue,
left_side, right_side) can be tested here, reusing Case 1's exact
labeled-example-probe CAV/TCAV machinery via the post-hoc classifier
wrapper.

Uses the better of the two architectures by post-hoc macro F1.

In [3]:
best_arch = max(case3_primary_results, key=lambda a: case3_primary_results[a]["test_metrics"]["macro_f1"])
print("best architecture by post-hoc macro F1:", best_arch)
wrapped = case3_models[best_arch]["wrapped"]

concept_results = run_concept_analysis(wrapped, train_loader, test_loader, device, class_names, CONCEPT_DEFINITIONS)
for concept, r in concept_results.items():
    print(f"{concept:12s} probe_acc={r['probe_accuracy']:.3f}  scores={ {k: round(v,3) for k,v in r['scores'].items()} }")

best architecture by post-hoc macro F1: Transformer


hand         probe_acc=0.990  scores={'baseline': 0.0, 'left_hand': 1.0, 'right_hand': 1.0, 'left_foot': 0.0, 'right_foot': 0.0, 'tongue': 0.0}
foot         probe_acc=0.989  scores={'baseline': 0.0, 'left_hand': 0.0, 'right_hand': 0.0, 'left_foot': 1.0, 'right_foot': 1.0, 'tongue': 0.0}
tongue       probe_acc=0.992  scores={'baseline': 0.0, 'left_hand': 0.0, 'right_hand': 1.0, 'left_foot': 0.0, 'right_foot': 0.0, 'tongue': 1.0}
right_side   probe_acc=0.993  scores={'baseline': 0.0, 'left_hand': 0.0, 'right_hand': 1.0, 'left_foot': 0.0, 'right_foot': 1.0, 'tongue': 0.0}
left_side    probe_acc=0.993  scores={'baseline': 1.0, 'left_hand': 1.0, 'right_hand': 0.0, 'left_foot': 1.0, 'right_foot': 0.0, 'tongue': 1.0}


### Cross-class rank-bootstrap significance test

For each concept, tests whether its CAV direction specifically implicates
its own intended target class more than the other 5 — not just whether it
scores highly for that class in isolation — across 1000 subject-level
bootstrap resamples of the test set. See
[`population-level-evaluation-plan.md`](../docs/population-level-evaluation-plan.md)
§6 for why this replaced a random-direction null (which turned out not to
be well-calibrated: within-class gradients are consistent enough that
almost any fixed direction scores near-extreme for a given class,
independent of the direction's semantic content).

In [4]:
train_features, train_labels = extract_pooled_features(wrapped, train_loader, device)
test_features, test_labels, test_subjects = extract_pooled_features_with_subjects(wrapped, test_loader, device)

# one target class per concept - the class the concept is defined to isolate
concept_target_class = {
    "hand": None,  # spans 2 classes (left_hand, right_hand); tested per-class below
    "foot": None,  # spans 2 classes (left_foot, right_foot); tested per-class below
    "tongue": class_names.index("tongue"),
    "right_side": None,  # spans 2 classes; tested per-class below
    "left_side": None,  # spans 2 classes; tested per-class below
}
# for concepts whose positive set spans >1 class, test rank-bootstrap
# separately against each of its positive classes
concept_positive_classes = {name: sorted(pos) for name, (pos, neg) in CONCEPT_DEFINITIONS.items()}

bootstrap_results = {}
t0 = time.time()
for concept_name, (positive, negative) in CONCEPT_DEFINITIONS.items():
    cav = train_cav(train_features, train_labels, positive, negative)
    per_target = {}
    for target_class in concept_positive_classes[concept_name]:
        result = cross_class_rank_bootstrap_test(
            wrapped, test_features, test_labels, test_subjects, cav["direction"],
            target_class=target_class, class_names=class_names, device=device,
            n_resamples=1000, seed=42,
        )
        per_target[class_names[target_class]] = result
        print(f"{concept_name:12s} -> {class_names[target_class]:11s} "
              f"TCAV={result['mean_tcav']:.3f} CI={result['ci_95']} P(rank1)={result['p_rank1']:.3f}")
    bootstrap_results[concept_name] = per_target
print(f"\n{time.time()-t0:.0f}s elapsed")

hand         -> left_hand   TCAV=1.000 CI=[1.0, 1.0] P(rank1)=0.507


hand         -> right_hand  TCAV=1.000 CI=[1.0, 1.0] P(rank1)=0.493


foot         -> left_foot   TCAV=1.000 CI=[1.0, 1.0] P(rank1)=0.507


foot         -> right_foot  TCAV=1.000 CI=[1.0, 1.0] P(rank1)=0.493


tongue       -> tongue      TCAV=1.000 CI=[1.0, 1.0] P(rank1)=0.493


right_side   -> right_hand  TCAV=1.000 CI=[1.0, 1.0] P(rank1)=0.507


right_side   -> right_foot  TCAV=1.000 CI=[1.0, 1.0] P(rank1)=0.493


left_side    -> left_hand   TCAV=1.000 CI=[1.0, 1.0] P(rank1)=0.245


left_side    -> left_foot   TCAV=1.000 CI=[1.0, 1.0] P(rank1)=0.257

50s elapsed


## Save results

In [5]:
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
summary = {
    "n_subjects": 100,
    "best_architecture": best_arch,
    "primary_results": {
        arch: {"best_val_contrastive_loss": r["best_val_contrastive_loss"], "test_macro_f1": r["test_metrics"]["macro_f1"]}
        for arch, r in case3_primary_results.items()
    },
    "concept_analysis": {
        name: {"probe_accuracy": r["probe_accuracy"], "scores": r["scores"]}
        for name, r in concept_results.items()
    },
    "bootstrap_rank_test": bootstrap_results,
}
results_path = RESULTS_DIR / "case3_validation_results.json"
with open(results_path, "w") as f:
    json.dump(summary, f, indent=2)
print("saved:", results_path)

saved: /Users/srinivasgovindasurampudi/Projects/neurolens-rag/results/case3_validation_results.json


## Summary

Case 3 (self-supervised brain-HRF co-embedding, same-window pairing, no
class label used anywhere in training) validated end-to-end on the current
100-subject dataset:

- Both GRU and Transformer backbones train successfully under the
  symmetric in-batch-negative contrastive loss.
- The post-hoc linear-probe classifier gives a macro-F1 directly comparable
  to Case 1 (92.0%) and Case 2 (91.8%) — see the printed primary-run output
  above for the actual measured numbers, and `results/case3_validation_results.json`
  for the full record.
- The `BrainWithPostHocClassifier` wrapper lets Case 1's existing
  `concepts.py` CAV/TCAV machinery run on Case 3's self-supervised
  representation completely unchanged.
- Fixed (closed-vocabulary) concept testing across the same 5 label-derived
  concepts used for Case 1/2, with the cross-class rank-bootstrap
  significance test (1000 subject-level resamples per concept-target pair)
  applied for the first time as a *built-in* step of a validation notebook,
  not just an ad hoc post-hoc analysis.

This validates the pipeline is ready to be queued into the full 3-case x
2-architecture x capacity-variant sweep once the 200-subject data scale-up
(task #56) completes, per the project's established prototype-then-scale
workflow.